# Handling Data for Nepali Post Sentiment Analysis

Following datasets will be used for sentiment analysis of emotions in different posts online. 
- https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
- https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv
- https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_3.csv

This dataset is choosen to let model understand different emotion via texts. This will allow us to get information on emotion or sentiment of human writing a post. Also the data from reddit where Nepali people posts will be collected and categorized and analysed using KMeans then again feed to the senitment analysis to get overall picture of nepali people's discussion and their emotions regarding it.


## Download GoEmotions dataset

### import libraries 
- request
- pandas

In [26]:
import requests
import pandas as pd 
import os

In [27]:
go_emotion_base_url = "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/"

# go emotion datasets names
go_emotion_datasets_names = [
    "goemotions_1.csv",
    "goemotions_2.csv",
    "goemotions_3.csv"
]

# make an empty list to append loaded pd
go_emotions_list = []

# use loop for goemotion
for goemotion in go_emotion_datasets_names:
    df = pd.read_csv(f"{go_emotion_base_url}{goemotion}")
    go_emotions_list.append(df)
    print(f"{goemotion} Loaded!")
    
go_emotions_dataset = pd.concat(go_emotions_list, ignore_index=True)

print(f"SHAPE OF FINAL GO_EMOTION DATASET = {go_emotions_dataset.shape}")
go_emotions_dataset.head(3)

goemotions_1.csv Loaded!
goemotions_2.csv Loaded!
goemotions_3.csv Loaded!
SHAPE OF FINAL GO_EMOTION DATASET = (211225, 37)


,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1


## Clean GoEmotions Data as per Needed:
- remove id, author, subreddit, link_id, parent_id, created_utc and rater_id, example_very_unclear
- clean text column (text preprocessing)

In [28]:
# drop unnecessary columns
go_emotions_dataset.drop(columns=[
    "id", "author", "subreddit", "link_id", 
    "parent_id", "created_utc", "rater_id", 
    "example_very_unclear"
], inplace=True)

print("Dropped unnecessary columns in place!")

Dropped unnecessary columns in place!


### Clean text requires additional libraries:
- regex
- bs4 BeautifulSoup
- string

Create a function to clean text usable in future as well. This function will remove html, punctuations, urls, additonal spaces, emojis, and also lowercase the text. It is required as a basic pre-processing for TF-IDF NLP.

In [29]:
import re
import string
from bs4 import BeautifulSoup
import emoji


# function to clean and direct apply
def clean_text(text):
    # remove html
    text = BeautifulSoup(text, "html.parser").get_text()
    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)
    # remove emoji
    text = emoji.replace_emoji(text, "")
    # return
    return text.lower().strip()

In [30]:
# Apply the function to text column
go_emotions_dataset["text"] = go_emotions_dataset["text"].apply(clean_text)

# See few texts how  they look
go_emotions_dataset.head()

,text,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,that game hurt,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,sexuality shouldnt be a grouping category it m...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,you do right if you dont care then fuck em,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,man i love reddit,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,name was nowhere near them he was by the falcon,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


### Drop null values if any and Save

In [31]:
# Drop na value and save
go_emotions_dataset = go_emotions_dataset.dropna(subset=["text"]).reset_index(drop=True)

go_emotions_dataset.isna().sum()

text              0
admiration        0
amusement         0
anger             0
annoyance         0
approval          0
caring            0
confusion         0
curiosity         0
desire            0
disappointment    0
disapproval       0
disgust           0
embarrassment     0
excitement        0
fear              0
gratitude         0
grief             0
joy               0
love              0
nervousness       0
optimism          0
pride             0
realization       0
relief            0
remorse           0
sadness           0
surprise          0
neutral           0
dtype: int64

In [33]:
# Save to the datasets folder
go_emotions_dataset.to_csv("./datasets/go_emotions.csv", index=False)

## Download YT Comments
This data will be downloaded googles Youtube API key. This dataset will contain nepali's voice including youths. The comments represent what nepali people think on current government, system, justice, education, etc. This shows if citizens are happy or not in different aspects of the country.

### Import Libraries
- build from googleapiclient.discovery
- dotenv
- LangDetectException

In [7]:
from dotenv import load_dotenv
from googleapiclient.discovery import build
from langdetect import detect, LangDetectException

# config dotenv
load_dotenv("./.env")

True

In [8]:
# youtube api key access
youtube = build(
    "youtube",
    "v3",
    developerKey=os.getenv("YT_API_KEY")
)

### Extracts channel ID
The channel id is not usually present like it used to, so it is necessary to extract the channel's id before extracting contents from the Api.

In [9]:
def get_channel_id(channel):

    # Already a channel ID
    if channel.startswith("UC"):
        return channel


    # Handle (@name)
    if channel.startswith("@"):

        response = youtube.channels().list(
            part="id",
            forHandle=channel.replace("@","")
        ).execute()


        if not response.get("items"):
            raise Exception(
                f"Channel not found: {channel}"
            )

        return response["items"][0]["id"]

    raise Exception(
        f"Invalid channel format: {channel}"
    )

### Selection of Channels
Only few channels are selected:
- TechTanka
- WhySoOffend
- Thaharesearch
- eon_visuals
- TheNepaliComment

These channels are popular in the youtube explaining current problems and news of Nepal. They also talk about: politics, corruption, system, justice, etc which brings engaging viewrs on the video who comment and share their point of view.

In [10]:
channels = [
    get_channel_id("@TechTanka"),
    get_channel_id("@WhySoOffended"),
    get_channel_id("@Thaharesearch"),
    get_channel_id("@eon_visuals"),
    get_channel_id("@TheNepaliComment")
]

### Extract playlists and recent videos

In [11]:
def get_upload_playlist(channel_id):

    response = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    ).execute()
    return response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

In [12]:
def get_recent_videos(
    channel_id,
    start_date,
    end_date,
    limit=10
):
    response = youtube.search().list(
        part="snippet",
        channelId=channel_id,
        order="date",
        maxResults=limit,
        type="video",
        publishedAfter=start_date,
        publishedBefore=end_date
    ).execute()

    videos=[]

    for item in response.get("items", []):

        videos.append({
            "video_id": item["id"]["videoId"],
            "title": item["snippet"]["title"],
            "date": item["snippet"]["publishedAt"]
        })

    return videos

### Recieve Comments
This function is the main function used internally to fetch all the comments in the video. 

In [13]:
def get_comments(video_id, limit=500):

    comments=[]
    
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )

    while request and len(comments) < limit:
        try:
            response = request.execute()
        except:
            break

        for item in response["items"]:
            snippet = (
                item["snippet"]
                ["topLevelComment"]
                ["snippet"]
            )

            comments.append({
                "comment": snippet["textDisplay"],
                "published_at": snippet["publishedAt"]
            })

        token = response.get("nextPageToken")

        if token:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                pageToken=token,
                maxResults=100,
                textFormat="plainText"
            )
        else:
            break
        
    return comments

### Quality control of Comments
This function only passes the quality comments. The quality control must fullfill following criteria for passing:
- Language must be English (NO-Devnagari)
- Length of raw text must be above 5
- Remove low value adding phrases

No-devnagari because devnagari will add additional challange to preprocess, meaning extraction and also creates larger groups for model's embedding to understand. Also there are limitations on semantic datasets for devnagari (NEPALI). This will also affect K-Means Clustering as the model might just seperate Devnagari embedding to a seperate chunk not giving approprite meaning with english embeddings.

Length of text must be above 5 because raw text too short might give lower semantic value. This does not give detail emotion on how person relates and feels, a simple " I have done this ..... and ..... which caused me to be depressed." structured is needed instead of "I am very sad." For model to generalize well.

Removal of useless phrases simply means removing comments that contains low value adding texts such has : nice video, wow, upload more, etc. as this does not align with the project's problem and rather focus on what people are talking about video not problem.

In [14]:
def is_quality_comment(text):
    try:
        if detect(text) != "en":
            return False
        
    except LangDetectException:
        return False

    if len(text.split()) < 10:
        return False

    low=text.lower().strip()

    useless = [
    # talking about the video quality itself
    "nice video",
    "great video",
    "awesome video",
    "amazing video",
    "excellent video",
    "good video",
    "very good video",
    "best video",
    "cool video",
    "beautiful video",
    "wonderful video",
    "fantastic video",
    "brilliant video",
    "incredible video",
    "interesting video",
    
    # talking about content without adding information
    "nice content",
    "great content",
    "awesome content",
    "amazing content",
    "good content",
    "best content",
    "love this content",
    "loved this content",

    # generic creator appreciation
    "thanks for the video",
    "thank you for the video",
    "thanks for uploading",
    "thanks for sharing",
    "thank you for sharing",
    "thanks for making this",
    "appreciate the video",
    "appreciate this video",

    # generic reactions
    "nice one",
    "good one",
    "great one",
    "awesome one",
    "well done",
    "good job",
    "great job",
    "awesome job",

    # Engagement spam
    "first",
    "first comment",
    "early",
    "early gang",
    "who is watching in 2026",
    "anyone here in 2026",
    "still watching",
    "watching in 2026",

    # Requests without discussion
    "make more videos",
    "more videos please",
    "upload more",
    "keep uploading",
    "keep making videos",
    "keep it up",
]

    for phrase in useless:
        if phrase in low:
            return False

    return True

### The main Scrape Function
Extracts video -> Extracts Comment -> Checks Quality -> Use/Throw

In [15]:
def scrape_channels(channels, start_date, end_date, limit):

    data=[]

    for channel_id in channels:
        print("CHANNEL:", channel_id)
        videos=get_recent_videos(
            channel_id,
            start_date,
            end_date,
            limit
        )
        
        for video in videos:
            print(
                "VIDEO:",
                video["title"]
            )
            comments=get_comments(
                video["video_id"]
            )
            for comment in comments:
                if is_quality_comment(comment["comment"]):
                    data.append({
                        "channel":channel_id,
                        "video":video["title"],
                        "video_id":video["video_id"],
                        "comment":comment["comment"],
                        "published_at":comment["published_at"]
                    })

    return pd.DataFrame(data)

In [16]:
yt_comments = scrape_channels(channels, "2024-01-01T00:00:00Z", "2026-07-25T00:00:00Z", 80)
yt_comments.head()

CHANNEL: UCJUFkLE5GjqloVigynCN4VA
VIDEO: The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता ! #TankaDahal #balenshah
VIDEO: सिस्टम बलियो बने देश बलियो बन्छ तर व्यक्ति बलियो बने तानाशाह जन्मिन्छ
VIDEO: एउटा लोकप्रिय नेताको ताना*शाह तर्फको यात्रा | Tanka Dahal
VIDEO: Why am I leaving Microsoft ? Something big is coming... | Tanka Dahal
VIDEO: The Truth : नेपालले जमिन मिचेकै हो त?  Border | India Nepal | Tanka Dahal
VIDEO: बोल्यो कि पोल्यो : बालेन साहको भाषणमा के छ त्यस्तो? । Tanka Dahal
VIDEO: The Truth : रिपोर्टमा के छ ? Media Trial VS Reality... | Tanka Dahal
VIDEO: गैर आवासीय नागरिकताको बास्तबिकता : Watch before you get NRN Citizenship ! [Podcast ]
VIDEO: अदालतमा राजनीतिको जालोको नालीबेली  ! Tanka Dahal
VIDEO:  अध्यादेश : बाध्यता, पेलान कि आवस्यकता ? [Analysis] | Tanka Dahal
VIDEO: सुकुम्बासी समस्या कि खोला किनारको राजनीति ? (Reupload)
VIDEO: सिंगापुरमा यस्तो, नेपालमा कस्तो? बालेन बन्लान त ली क्वान? | Tanka Dahal #singapore #nepal
VIDEO: Truth behind the report... | Tanka Dahal
VIDEO: [Pa

,channel,video,video_id,comment,published_at
0,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,"Thank you,plz support country. Not person who ...",2026-07-21T05:24:45Z
1,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,Thank you uncle for your insights. \nMy indepe...,2026-07-20T15:51:06Z
2,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,open your eyes and mind don't be manipulated b...,2026-07-20T14:01:02Z
3,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,Nagar prahari lai training dinu parcha they al...,2026-07-20T04:48:59Z
4,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,How does one man surpass the nation's media? \...,2026-07-20T02:09:24Z


In [17]:
yt_comments["comment"] = yt_comments["comment"].apply(clean_text)

print(f"Total Length of data = {yt_comments.shape}")

yt_comments.head()

Total Length of data = (8843, 5)


,channel,video,video_id,comment,published_at
0,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,thank youplz support country not person who wa...,2026-07-21T05:24:45Z
1,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,thank you uncle for your insights my independe...,2026-07-20T15:51:06Z
2,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,open your eyes and mind dont be manipulated by...,2026-07-20T14:01:02Z
3,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,nagar prahari lai training dinu parcha they al...,2026-07-20T04:48:59Z
4,UCJUFkLE5GjqloVigynCN4VA,The Truth : चमे*ली ग्यां*ग - भ्रम र वास्तविकता...,Cyi4xfHYswQ,how does one man surpass the nations media by ...,2026-07-20T02:09:24Z


In [18]:
yt_comments.to_csv("./datasets/yt_comments.csv", index=False)